# Geometric steering

**Steer like a driver looks ahead.**

Nobody drives a bend by staring at the white line under the bonnet.
You look where the road is going, a second or so ahead, and steer to get there.
Unit 4's controller does the opposite: it reacts to the cross-track error and the heading error **now**.
On racekit's perfect kinematic car that works beautifully.

Here is the same Unit 4 agent at 30 m/s on `test-circuit`, twice.
The second car is identical except that its steering, throttle and brake take effect **two control ticks (0.1 s) late**, the way a real car's actuators do.

In [ ]:
import math
import sys
from pathlib import Path
from tempfile import mkdtemp

import matplotlib.pyplot as plt
import numpy as np
from racekit import write_variant
from racekit.cars import KinematicCar
from racekit.runner import load_submission, run_race, shim_installed
from racekit.tracks import load_track

AGENTS = Path("../../projects/racekit/agents")
STARTER, UNIT_04, UNIT_05 = AGENTS / "unit-01.py", AGENTS / "unit-04.py", AGENTS / "unit-05.py"
workdir = Path(mkdtemp(prefix="roar-unit-05-"))
TICK = 0.05 + 1e-9  # race times are pinned to one control tick
RMS_TOL = 0.0005  # m, the book's tolerance on an RMS cross-track error


def variant(source, name, **constants):
    # Unit 1's idiom: a copy of a submission with some module constants changed.
    return write_variant(source, workdir / f"{name}.py", {key: repr(value) for key, value in constants.items()})


def rms_offset(result):
    # RMS cross-track error over the driving rows (held == 0), as in Unit 4.
    telemetry = result.telemetry
    offset = telemetry["offset"][telemetry["held"] == 0]
    return float(np.sqrt(np.mean(offset ** 2)))


def clean(result):
    return result.finished and result.collisions == 0


def summary(result):
    if clean(result):
        return f"clean, {result.total_time:6.2f} s, RMS {rms_offset(result):.4f} m"
    finished = "finished" if result.finished else "did not finish"
    return f"{result.collisions:2d} collisions, {finished} at {result.total_time:.2f} s"


def delay_race(source, lag, seed=0, **constants):
    # The delay study's setting: test-circuit at 30 m/s, jitter 0.05, an actuator delay of `lag` ticks, 400 s cap.
    constants["TARGET_SPEED"] = 30.0
    name = "-".join([source.stem, f"lag{lag}"] + [f"{key}{value}" for key, value in sorted(constants.items())])
    car = KinematicCar(jitter=0.05, lag_ticks=lag)
    return run_race(variant(source, name, **constants), "test-circuit", car=car, seed=seed, max_seconds=400.0)


races = {}  # (controller, lag) -> RaceResult, all seed 0
races[("Unit 4 PD", 0)] = delay_race(UNIT_04, 0)
races[("Unit 4 PD", 2)] = delay_race(UNIT_04, 2)
for lag in (0, 2):
    print(f"Unit 4 at 30 m/s, actuator delay {lag} ticks:", summary(races[("Unit 4 PD", lag)]))
assert clean(races[("Unit 4 PD", 0)]) and races[("Unit 4 PD", 2)].collisions == 41

The controller that held the line to 0.13 m now hits the wall **41 times** in 400 s.
Nothing about the track, the speed or the gains changed; only *when* the commands arrive.

Every real car has some delay between a command and its effect, and CARLA's is unknown until Unit 8 measures it.
So the question of this unit is: **can a controller that looks ahead survive a delay that a controller that only reacts cannot?**

The answer is yes, and it comes with a price.
By the end of this lesson you will have three steering laws (pure pursuit, a speed-scaled lookahead and Stanley), a delay study that shows which survive, and an honest account of what the survivor costs when there is no delay.
That account decides what `agents/unit-05.py` ships, and the lab lets the real car have the last word.

## 1. Pure pursuit: the arc through a point ahead

Pick a **target point** on the path, some distance ahead of the car.
Of all the circular arcs that leave the car along its heading, exactly one passes through the target.
Pure pursuit steers the car onto that arc, and picks a new target every tick.

Work in the car frame (Unit 2's `to_car_frame`): the car is at the origin heading along +forward.
Let the target be at straight-line distance d and bearing α (left positive).
The arc is tangent to the heading at the car, so its centre lies on the car's left (or right) normal, at distance R.
The car, the target and the centre form an isosceles triangle with two sides R.
The angle between a tangent and a chord equals half the central angle the chord subtends (the tangent–chord angle), so the chord subtends 2α, and

$$d = 2R\sin\alpha \qquad\Longrightarrow\qquad \kappa = \frac{1}{R} = \frac{2\sin\alpha}{d}.$$

The sign comes for free: a target on the left (α > 0) gives κ > 0, a left turn, the same sign convention as Unit 2's Menger curvature.
Unit 3's kinematic bicycle turns that curvature into a wheel angle, κ = tan δ / L, so

$$\delta = \arctan(L\,\kappa) = \operatorname{atan2}(2L\sin\alpha,\; d), \qquad \text{steer} = -\delta / \delta_{\max}.$$

Two lines of code.
Before trusting them on a track, check them where the answer is known exactly: **a car on a circle**.
Put the car on a circle of radius R, tangent to it, and the target on the same circle at chord distance d.
The arc through the target *is* the circle, so pure pursuit must return κ = 1/R for every d, and δ = atan(L/R).

In [ ]:
WHEELBASE = 2.875  # m, L (Unit 3)
DELTA_MAX = math.radians(70.0)  # the wheel angle at |steer| = 1


def to_car_frame(px, py, cx, cy, yaw):
    # Unit 2: the world point (px, py) seen from a car at (cx, cy) with heading yaw, as (forward, left).
    dx, dy = px - cx, py - cy
    c, s = math.cos(yaw), math.sin(yaw)
    return c * dx + s * dy, -s * dx + c * dy


def pursuit_curvature(alpha, distance):
    # The arc that leaves the car along its heading and passes through a point at bearing alpha, distance d.
    return 2.0 * math.sin(alpha) / distance


def pursuit_wheel_angle(alpha, distance):
    # delta = atan(L * kappa), written as atan2 (the book's arctan).
    return math.atan2(2.0 * WHEELBASE * math.sin(alpha), distance)


def target_on_circle(cx, cy, yaw, radius, chord):
    # The point at straight-line distance `chord` along the circle of signed radius `radius` (left positive)
    # that is tangent to the car's heading at the car.
    half = chord / 2.0
    theta = 2.0 * math.atan2(half, math.sqrt(radius * radius - half * half)) * (1.0 if radius > 0 else -1.0)
    ox, oy = cx - radius * math.sin(yaw), cy + radius * math.cos(yaw)  # the centre
    return ox + radius * math.sin(yaw + theta), oy - radius * math.cos(yaw + theta)


car_x, car_y, car_yaw = 40.0, -15.0, 0.7  # any pose will do
worst_kappa = worst_delta = 0.0
for radius in (18.0, 25.0, 120.0, -18.0, -25.0, -120.0):
    for chord in (3.0, 6.0, 9.0, 12.0):
        tx, ty = target_on_circle(car_x, car_y, car_yaw, radius, chord)
        forward, left = to_car_frame(tx, ty, car_x, car_y, car_yaw)
        alpha, distance = math.atan2(left, forward), math.hypot(forward, left)
        worst_kappa = max(worst_kappa, abs(pursuit_curvature(alpha, distance) - 1.0 / radius))
        worst_delta = max(worst_delta, abs(pursuit_wheel_angle(alpha, distance) - math.atan2(WHEELBASE / radius, 1.0)))
    print(f"R = {radius:7.1f} m: kappa = 1/R = {1.0 / radius:+.5f} 1/m, delta = atan(L/R) = {math.degrees(math.atan2(WHEELBASE / radius, 1.0)):+6.2f} deg")
print(f"worst |kappa - 1/R| = {worst_kappa:.1e} 1/m, worst |delta - atan(L/R)| = {worst_delta:.1e} rad")
assert worst_kappa < 1e-12 and worst_delta < 1e-12

Exact, for every chord, for left and right turns.
That is the reason pure pursuit is a good idea: on a path of constant curvature, any target on the path gives the right steer, however far ahead it is.

The submission `agents/unit-05.py` has the same two functions, `pursuit_curvature` and `pursuit_wheel_angle`.
Load it and run the same check on its copies (a submission's module functions live on its module, which `load_submission` names):

In [ ]:
with shim_installed():
    Agent, module_name = load_submission(UNIT_05)
submission = sys.modules[module_name]

worst = 0.0
for radius in (18.0, 25.0, 120.0):
    for chord in (3.0, 6.0, 9.0, 12.0):
        tx, ty = target_on_circle(car_x, car_y, car_yaw, radius, chord)
        forward, left = to_car_frame(tx, ty, car_x, car_y, car_yaw)
        alpha, distance = math.atan2(left, forward), math.hypot(forward, left)
        worst = max(worst, abs(submission.pursuit_curvature(alpha, distance) - 1.0 / radius),
                    abs(submission.pursuit_wheel_angle(alpha, distance) - math.atan2(WHEELBASE / radius, 1.0)))
print(f"the submission's functions: worst error {worst:.1e}")
assert worst < 1e-12

## 2. Choosing the target, and why a chord cuts the corner

A real track is not one circle.
The submission picks its target the simplest way that works:

- the **lookahead distance** is L_d = clip(k·v, L_min, L_max), with k = `PP_K` = 0.3 s, L_min = 3 m and L_max = 40 m.
  It **grows with the speed**: the car looks 0.3 s ahead, 6 m at 20 m/s and 9 m at 30 m/s;
- the **target** is the first waypoint after the car's segment (Unit 2's `segment_index`) whose straight-line distance from the car is at least L_d.

Then α comes from `to_car_frame`, and the wheel angle from the two lines above.

On a straight, or on a circle, the arc lands on the path.
At a **corner entry** it does not: the target is already round the bend, so the arc starts turning before the corner does and the car runs **inside** it.
Here is the tightest corner of `test-circuit`, with a car on the centreline of the straight 4 m before the bend begins, heading along the track, and the arcs for three lookaheads:

In [ ]:
def menger_curvature(xs, ys):
    # Unit 2: the signed curvature at each waypoint from (i - 1, i, i + 1), cyclic; left positive.
    n = len(xs)
    before, after = (np.arange(n) - 1) % n, (np.arange(n) + 1) % n
    ax, ay, cx, cy = xs[before], ys[before], xs[after], ys[after]
    twice_area = (xs - ax) * (cy - ay) - (ys - ay) * (cx - ax)
    sides = np.hypot(xs - ax, ys - ay) * np.hypot(cx - xs, cy - ys) * np.hypot(cx - ax, cy - ay)
    return 2.0 * twice_area / sides


def first_target(xs, ys, segment, px, py, lookahead):
    # The submission's rule: the first waypoint after `segment` at least `lookahead` metres away.
    n = len(xs)
    k = segment
    for _ in range(n):
        k = (k + 1) % n
        if math.hypot(xs[k] - px, ys[k] - py) >= lookahead:
            return k
    return k


track = load_track("test-circuit")
xs, ys = track.x, track.y
kappa_track = menger_curvature(xs, ys)
corner = int(np.argmax(np.abs(kappa_track)))
n = len(xs)
entry = corner
while abs(kappa_track[(entry - 1) % n]) > 1e-3:  # walk back to where the bend begins
    entry -= 1
start = (entry - 2) % n  # waypoints are about 2 m apart: the car is 4 m before the bend
px, py = xs[start], ys[start]
yaw = math.atan2(ys[(start + 1) % n] - ys[start], xs[(start + 1) % n] - xs[start])
print(f"track curvature at the car: {kappa_track[start]:+.4f} 1/m; in the corner: {kappa_track[corner]:+.4f} 1/m (R = {1 / kappa_track[corner]:.1f} m)")
assert abs(kappa_track[start]) < 1e-6

fig, ax = plt.subplots(figsize=(7, 6))
window = np.arange(start - 5, start + 30) % n
ax.plot(xs[window], ys[window], "k-", linewidth=1.0, label="centreline")
for lookahead, colour in ((6.0, "tab:blue"), (12.0, "tab:orange"), (20.0, "tab:green")):
    k = first_target(xs, ys, start, px, py, lookahead)
    forward, left = to_car_frame(xs[k], ys[k], px, py, yaw)
    kappa = pursuit_curvature(math.atan2(left, forward), math.hypot(forward, left))
    s = np.linspace(0.0, math.hypot(forward, left), 50)  # about the arc length to the target
    arc_x = px + (np.sin(yaw + kappa * s) - math.sin(yaw)) / kappa
    arc_y = py - (np.cos(yaw + kappa * s) - math.cos(yaw)) / kappa
    ax.plot(arc_x, arc_y, color=colour, label=f"L_d = {lookahead:.0f} m: kappa = {kappa:+.4f} 1/m")
    ax.plot([xs[k]], [ys[k]], "o", color=colour)
ax.plot([px], [py], "ks")
ax.set_aspect("equal")
ax.set_title(f"test-circuit, the tightest corner (R = {1 / kappa_track[corner]:.0f} m), the car 4 m before it")
ax.legend()
plt.show()

Each arc is drawn from the car to its target (the dot).
The track is still straight under the car, yet every lookahead already asks for a turn, and a gentler one than the 18 m corner: the arcs leave the straight early and run inside the bend, and the longer the lookahead the earlier and gentler the turn.
The arc is a **chord** across the bend: the car cuts the inside of the corner on the way in and straightens early on the way out.
Unit 4's law has no such bias: its curvature feedforward uses the curvature *at the car's segment*, and its PD pulls the error to zero.

## 3. Pure pursuit on the track, with no delay

The honest first test is the unit's own setting: 20 m/s, no delay.
Race both submissions with no jitter (deterministic) on `test-circuit`, and look at the cross-track error.

In [ ]:
unit_04 = run_race(UNIT_04, "test-circuit", car=KinematicCar(jitter=0.0))
pursuit = run_race(UNIT_05, "test-circuit", car=KinematicCar(jitter=0.0))
for name, result in (("Unit 4 PD", unit_04), ("pure pursuit", pursuit)):
    driving = result.telemetry["held"] == 0
    print(f"{name:>12}: {summary(result)}, max |offset| {np.max(np.abs(result.telemetry['offset'][driving])):.3f} m")
assert clean(unit_04) and abs(unit_04.total_time - 400.55) <= TICK and abs(rms_offset(unit_04) - 0.0139) <= RMS_TOL
assert clean(pursuit) and abs(pursuit.total_time - 400.50) <= TICK and abs(rms_offset(pursuit) - 0.0193) <= RMS_TOL
assert rms_offset(pursuit) > rms_offset(unit_04)

fig, ax = plt.subplots(figsize=(11, 3.5))
lap = unit_04.telemetry["t"] < 135.0  # the first lap
ax.plot(unit_04.telemetry["t"][lap], unit_04.telemetry["offset"][lap], label="Unit 4 PD")
lap = pursuit.telemetry["t"] < 135.0
ax.plot(pursuit.telemetry["t"][lap], pursuit.telemetry["offset"][lap], label="pure pursuit")
ax.set_xlabel("t (s)")
ax.set_ylabel("cross-track error (m, left +)")
ax.set_title("test-circuit at 20 m/s, no jitter, no delay: the first lap")
ax.legend()
plt.show()

Both are clean and within a tick of each other in time, but pure pursuit sits **further off the line**: each corner shows up as a bump, on the inside.
With actuator jitter (σ = 0.05, seeds 0–4) the picture is the same on every track.
These are the probe medians (range in brackets) from the plan's record, which the milestone and the routed tests re-race:

| track | Unit 4 PD | pure pursuit, k = 0.3 s | Stanley (§7) |
|---|---|---|---|
| `oval` | 0.1041 (0.0062) | 0.1422 (0.0107) | 0.0987 |
| `test-circuit` | 0.1048 (0.0050) | 0.1481 (0.0050) | 0.1043 |
| `monza-like` | 0.1029 (0.0042) | 0.1445 (0.0022) | 0.0981 |

(RMS cross-track error in metres, all 45 races clean.)

So, said plainly: **on a car with no delay, pure pursuit does not beat Unit 4's controller.**
It cuts corners, and nothing on this car punishes Unit 4's tight gains.
If racekit's kinematic car were the real car, this unit would end here.

## 4. Actuator delay

A command takes time to become motion.
On a real car:

- the steering is turned by a motor with a **rate limit**: the wheels cannot jump to a new angle;
- the commands travel through a computer and a bus, and CARLA applies them **asynchronously** to its physics, so the action you send this tick acts during a later one;
- a **tyre** makes lateral force by slipping, and the slip takes time to build (Unit 6).

The simplest model of all of that is a pure **delay**: the action commanded at tick k takes effect at tick k + N.
`KinematicCar(lag_ticks=N)` does exactly that, with a first-in, first-out queue of N actions; the queue starts (and restarts after a respawn) full of zero actions.
Watch the heading of a car at 20 m/s that is told to steer from the very first tick:

In [ ]:
for lag in range(4):
    car = KinematicCar(lag_ticks=lag)
    car.reset(0.0, 0.0, 0.0, 20.0)
    yaws = np.zeros(6)
    for tick in range(6):
        car.advance(0.0, 0.1, 0.0, 0.0, 0.0, 0.05, 10)  # throttle, steer, brake, hand brake, reverse, dt, substeps
        yaws[tick] = car.yaw
    first = int(np.nonzero(yaws)[0][0]) + 1
    print(f"lag_ticks = {lag}: yaw after ticks 1-6 = {np.round(yaws, 4)}; first moves on tick {first}")
    assert first == lag + 1

With N ticks of delay the car does not respond at all for N ticks; at 30 m/s, two ticks are **3 m** driven blind.

How much delay should we study?
Lag 2 is the smallest delay that separates the controllers on the kinematic car (at lag 1 all four are clean, below), and it stands in for the real car's one-tick lag plus its steering-rate limit and tyre dynamics, which the kinematic car does not have.

Why should a delay hurt Unit 4's controller in particular?
Its D term is the heading error, ė = −v·sin ψ_e (Unit 4), so K_HEAD = K_D·v: at a fixed K_HEAD the effective damping **falls as 1/v**, a third lower at 30 m/s than at 20.
And a delay acts like a loss of damping: the controller keeps pushing in the direction that was right 0.1 s ago.

## 5. The delay study

The setting: `test-circuit`, `TARGET_SPEED = 30.0`, jitter 0.05, seed 0, a 400 s cap, and lags 0 to 3.
Four controllers: the starter (with its P speed loop), Unit 4's PD, Stanley (§7) and pure pursuit with k = 0.3 s; the last three share Unit 4's PI speed loop.
Every one is a `write_variant` of a reference submission: `STEERING = "stanley"` turns `agents/unit-05.py` into the Stanley agent.

The crash races run to the 400 s cap, so this cell takes about a minute.
The starter at lag 2 is left to the milestone (it does not finish inside the cap, about 12 s of racing).

In [ ]:
CONTROLLERS = {
    "starter": (STARTER, {}),
    "Unit 4 PD": (UNIT_04, {}),
    "Stanley": (UNIT_05, {"STEERING": "stanley"}),
    "pure pursuit": (UNIT_05, {}),
}
for lag in range(4):
    for name, (source, constants) in CONTROLLERS.items():
        if (name, lag) in races or (name, lag) == ("starter", 2):
            continue
        races[(name, lag)] = delay_race(source, lag, **constants)

for lag in range(4):
    print(f"lag {lag}:")
    for name in CONTROLLERS:
        print(f"  {name:>12}: {summary(races[(name, lag)]) if (name, lag) in races else '(raced in the milestone)'}")

Read it column by column.

- **Lag 0:** everyone is clean and Unit 4's PD is the best line-follower; pure pursuit is the worst at 0.31 m, more than twice the PD's RMS, because at 30 m/s it looks 9 m ahead and cuts every corner.
- **Lag 1:** everyone is still clean and every RMS has grown: the PD's by about two thirds, pure pursuit's by about an eighth.
- **Lag 2:** Unit 4's PD and Stanley hit the wall over and over, while pure pursuit is **clean**, at 0.46 m.
- **Lag 3:** only pure pursuit survives, and it weaves (about 13 s slower, RMS 1.43 m): clean is not the same as good.

These are seed-0 races, pinned to the plan's probe record:

In [ ]:
PINS = {  # (controller, lag): (collisions, finished, time, RMS); None where a crash makes it meaningless
    ("starter", 0): (0, True, 279.65, 0.2447), ("Unit 4 PD", 0): (0, True, 268.70, 0.1306),
    ("Stanley", 0): (0, True, 268.55, 0.1575), ("pure pursuit", 0): (0, True, 268.75, 0.3097),
    ("starter", 1): (0, True, 280.25, 0.3335), ("Unit 4 PD", 1): (0, True, 269.40, 0.2150),
    ("Stanley", 1): (0, True, 269.00, 0.1904), ("pure pursuit", 1): (0, True, 269.05, 0.3498),
    ("Unit 4 PD", 2): (41, False, None, None), ("Stanley", 2): (20, False, None, None),
    ("pure pursuit", 2): (0, True, 269.70, 0.4559),
    ("starter", 3): (41, False, None, None), ("Unit 4 PD", 3): (60, False, None, None),
    ("Stanley", 3): (53, False, None, None), ("pure pursuit", 3): (0, True, 282.00, 1.4305),
}
for key, (collisions, finished, time, rms) in PINS.items():
    result = races[key]
    assert result.collisions == collisions and result.finished == finished, (key, result)
    if time is not None:
        assert abs(result.total_time - time) <= TICK and abs(rms_offset(result) - rms) <= RMS_TOL, (key, result)
print(len(PINS), "seed-0 races match the probe record")

One seed is one draw of the jitter, so the plan raced seeds 0–4 as well.
The five-seed record (RMS median and range in metres, or the per-seed collisions; the milestone and a routed test re-race the lag-0 and lag-2 rows):

| lag | Unit 4 PD | Stanley | pure pursuit, k = 0.3 s | starter |
|---|---|---|---|---|
| 0 | clean, 0.1295 (0.0060) | clean, 0.1582 (0.0139) | clean, 0.3097 (0.0153) | clean, 0.2346 (0.0167) |
| 1 | clean, 0.2136 (0.0077) | clean, 0.1939 (0.0129) | clean, 0.3469 (0.0198) | clean, 0.3233 (0.0154) |
| 2 | crashes [41, 42, 43, 43, 40] | crashes [20, 16, 16, 17, 20] | **clean**, 0.4526 (0.0234) | does not finish in 400 s |
| 3 | crashes, 60 each | crashes [53, 53, 53, 53, 50] | clean, 1.4305 (0.1659) | crashes [41, 41, 41, 41, 44] |

The pattern holds on every seed: **at lag 2, the speed-scaled lookahead keeps the car stable where the tighter controllers go unstable, and at lag 0 it pays for that by cutting corners.**
That is the trade this unit is named after, stability against corner-cutting.

Look at how the PD fails.
Its steer at lag 0 and at lag 2, and pure pursuit's at lag 2, over the first 20 s:

In [ ]:
fig, ax = plt.subplots(figsize=(11, 3.5))
for key in (("Unit 4 PD", 0), ("Unit 4 PD", 2), ("pure pursuit", 2)):
    telemetry = races[key].telemetry
    early = telemetry["t"] < 20.0
    ax.plot(telemetry["t"][early], telemetry["steer"][early], label=f"{key[0]}, lag {key[1]}")
ax.set_xlabel("t (s)")
ax.set_ylabel("steer command")
ax.set_title("test-circuit at 30 m/s: the commanded steer")
ax.legend()
plt.show()

At lag 2 the PD's steer **oscillates with a growing amplitude** until the car leaves the track: each correction arrives 0.1 s late, overshoots, and the next correction is larger.
That is an instability, not a matter of a little more error.
Pure pursuit's steer at the same lag stays smooth.

## 6. The lookahead, swept

If looking ahead is what saves pure pursuit, how far ahead?
Sweep `PP_K` ∈ {0.2, 0.3, 0.4, 0.5} s (L_d = 6, 9, 12, 15 m at 30 m/s), at lag 0 and lag 2, on the same seed.
All of these races are clean, so the cell is quick.

In [ ]:
LOOKAHEADS = (0.2, 0.3, 0.4, 0.5)
sweep = {}
for lag in (0, 2):
    for k in LOOKAHEADS:
        sweep[(k, lag)] = races[("pure pursuit", lag)] if k == 0.3 else delay_race(UNIT_05, lag, PP_K=k)

SWEEP_PINS = {  # seed 0: (time, RMS)
    (0.2, 0): (268.70, 0.1823), (0.3, 0): (268.75, 0.3097), (0.4, 0): (268.80, 0.4631), (0.5, 0): (268.75, 0.6477),
    (0.2, 2): (294.35, 1.3479), (0.3, 2): (269.70, 0.4559), (0.4, 2): (269.50, 0.5677), (0.5, 2): (269.45, 0.7203),
}
for key, (time, rms) in SWEEP_PINS.items():
    result = sweep[key]
    print(f"PP_K = {key[0]} s, lag {key[1]}: {summary(result)}")
    assert clean(result) and abs(result.total_time - time) <= TICK and abs(rms_offset(result) - rms) <= RMS_TOL

best = {lag: min(LOOKAHEADS, key=lambda k: rms_offset(sweep[(k, lag)])) for lag in (0, 2)}
print("the best lookahead:", best)
assert best == {0: 0.2, 2: 0.3}

fig, ax = plt.subplots(figsize=(6, 4))
for lag in (0, 2):
    ax.plot(LOOKAHEADS, [rms_offset(sweep[(k, lag)]) for k in LOOKAHEADS], "o-", label=f"lag {lag}")
ax.set_xlabel("PP_K (s): the lookahead is PP_K * v")
ax.set_ylabel("RMS cross-track error (m)")
ax.set_title("pure pursuit, test-circuit at 30 m/s, seed 0")
ax.legend()
plt.show()

At lag 0 the shortest lookahead is best and every extra 3 m of lookahead costs another 0.13–0.18 m of RMS: pure corner-cutting.
At lag 2 the shortest lookahead **weaves** (clean, but 25 s slower and 1.35 m off the line), and the minimum moves to k = 0.3 s.
**The best lookahead grows with the delay.**
The five-seed medians agree (lag 0: 0.1800, 0.3097, 0.4656, 0.6477 m; lag 2: 1.2972, 0.4526, 0.5625, 0.7203 m; all 40 races clean).

Why does a longer lookahead help?
Linearize pure pursuit about a straight line.
A car at cross-track error e (left positive) with a heading ψ relative to the line sees a target L_d ahead on the line at lateral position about −e − L_d·ψ in its own frame, so sin α ≈ −(e + L_d ψ)/L_d and

$$\delta \approx L\kappa = -\frac{2L}{L_d^2}\,e \;-\; \frac{2L}{L_d}\,\psi .$$

That is **a PD law** with K_E = 2L/L_d² and K_HEAD = 2L/L_d (ψ_e = −ψ in Unit 4's notation).
Because L_d = k·v, its gains **fall as the car speeds up**: pure pursuit retunes itself with the speed.

In [ ]:
print(f"{'v (m/s)':>8}{'L_d (m)':>9}{'K_E (rad/m)':>13}{'K_HEAD':>9}")
equivalent = {}
for v in (20.0, 30.0):
    lookahead = min(max(0.3 * v, 3.0), 40.0)
    equivalent[v] = (2.0 * WHEELBASE / lookahead ** 2, 2.0 * WHEELBASE / lookahead)
    print(f"{v:8.0f}{lookahead:9.1f}{equivalent[v][0]:13.4f}{equivalent[v][1]:9.3f}")
print("Unit 4's gains: K_E = 0.2, K_HEAD = 1.0 at every speed")

# check the linearization against the exact law for a small error
e, psi, lookahead = 0.05, 0.01, 9.0
forward, left = to_car_frame(lookahead, 0.0, 0.0, e, psi)  # target on the line y = 0, about L_d ahead
exact = pursuit_wheel_angle(math.atan2(left, forward), math.hypot(forward, left))
linear = -2.0 * WHEELBASE / lookahead ** 2 * e - 2.0 * WHEELBASE / lookahead * psi
print(f"exact delta {exact:.6f} rad, linearized {linear:.6f} rad")
assert abs(exact - linear) < 2e-4
assert abs(equivalent[30.0][0] - 0.0710) < 1e-4 and abs(equivalent[30.0][1] - 0.639) < 1e-3

At 20 m/s pure pursuit behaves like a PD about as tight as Unit 4's; at 30 m/s like one with a third of Unit 4's K_E and two thirds of its K_HEAD.
A softer controller corrects less sharply, so the late corrections that a delay causes do not snowball.
(The linearization puts the target exactly L_d ahead on the line; the submission's first-waypoint rule lands it up to one waypoint spacing further, so on the track the effective gains are a little softer still.)

## 7. Stanley, and a detuned PD

**Stanley** (the steering law of Stanford's car Stanley, which won the 2005 DARPA Grand Challenge) steers from the **front axle**.
Project the front-axle point (x + L cos ψ, y + L sin ψ) onto the path to get its cross-track error e_f (left positive), and take the segment's heading error ψ_e = wrap(segment heading − ψ):

$$\delta = \psi_e + \operatorname{atan2}\!\left(-k\,e_f,\; \max(v, 1)\right), \qquad k = \texttt{ST\_K} = 3\ \text{s}^{-1}.$$

The first term points the wheels along the path; the second steers the front axle back to it, more gently the faster the car goes (the `max(v, 1)` keeps it defined at rest).
Here it is on one pose, a car 0.5 m left of a straight line along +x, heading along it:

In [ ]:
def stanley_wheel_angle(px, py, yaw, speed, k=3.0):
    # Stanley on the straight line y = 0 (heading 0): the front axle's offset is its y.
    front_y = py + WHEELBASE * math.sin(yaw)
    heading_error = (0.0 - yaw + math.pi) % (2.0 * math.pi) - math.pi
    return heading_error + math.atan2(-k * front_y, max(speed, 1.0))


for speed in (10.0, 20.0, 30.0):
    delta = stanley_wheel_angle(0.0, 0.5, 0.0, speed)
    print(f"v = {speed:4.0f} m/s: delta = {delta:+.4f} rad ({'right' if delta < 0 else 'left'}, back toward the line)")
assert stanley_wheel_angle(0.0, 0.5, 0.0, 20.0) < 0.0

With no delay Stanley is excellent: the best of the three at 20 m/s (the §3 table) and second only to Unit 4's PD at 30 m/s.
But it has **no lookahead**: it reacts to where the front axle is now, with an effective heading gain of about 1 + k·L/v ≈ 1.3 at 30 m/s and a cross-track gain k/v = 0.1 rad/m, as tight as Unit 4's.
At lag 2 it crashes on every seed (§5).

So is the lesson "look ahead", or just "be gentle"?
Detune Unit 4's PD by hand, K_E = 0.05 rad/m and K_HEAD = 0.5, and race it in the delay study's setting at lag 0 and lag 2:

In [ ]:
detuned = {lag: delay_race(UNIT_04, lag, K_E=0.05, K_HEAD=0.5) for lag in (0, 2)}
for lag in (0, 2):
    print(f"lag {lag}: detuned PD {summary(detuned[lag])};  pure pursuit {summary(races[('pure pursuit', lag)])}")
assert clean(detuned[0]) and abs(detuned[0].total_time - 268.90) <= TICK and abs(rms_offset(detuned[0]) - 0.3549) <= RMS_TOL
assert clean(detuned[2]) and abs(detuned[2].total_time - 269.75) <= TICK and abs(rms_offset(detuned[2]) - 0.5174) <= RMS_TOL
for lag in (0, 2):
    assert rms_offset(races[("pure pursuit", lag)]) < rms_offset(detuned[lag])

The detuned PD **survives lag 2 too**: being gentle is enough to stay on the track.
But it sits further off the line than pure pursuit at both lags.
Over five seeds (the plan's record, re-raced by the milestone) the gap is beyond the noise: at lag 2, pure pursuit 0.4526 m (range 0.0234) against the detuned PD's 0.5149 m (range 0.0376).

Pure pursuit gets that detuning **for free**, from its lookahead, and it gets the *right amount* at every speed (§6), while a hand-detuned PD is detuned at 20 m/s too, where it did not need to be.
That, not a better line, is why the geometric law is the robust choice when the delay is unknown.

## 8. Choosing under uncertainty

Put the evidence side by side.

| on racekit's kinematic car | no delay | 2-tick delay at 30 m/s |
|---|---|---|
| Unit 4's PD | best line (RMS 0.10 m at 20 m/s, 0.13 m at 30 m/s) | crashes on every seed |
| Stanley | as good as the PD | crashes on every seed |
| a hand-detuned PD | clean, well off the line | clean, 0.51 m |
| pure pursuit, k = 0.3 s | cuts corners (0.14 m at 20 m/s, 0.31 m at 30 m/s) | **clean, 0.45 m** |

Which one should the submission use?
That depends on a number we do not have yet: **the real car's delay**.
If CARLA's car behaves like lag 0, Unit 4's controller follows the line more closely; if it behaves like lag 2 or worse, Unit 4's controller can put it in the wall.
The cost of the wrong choice is not symmetric: a few centimetres of line against a crash (and Unit 11 will show you what one crash costs).

So `agents/unit-05.py` ships **pure pursuit**, `STEERING = "pursuit"`, with `PP_K = 0.3`, and keeps Unit 4's law one constant away: with `STEERING = "pid"` the file races exactly like `agents/unit-04.py`, to the tick (the milestone checks it).
`TARGET_SPEED` stays 20 m/s; going faster is Units 6 and 7.

This is a decision, not a proof.
**The lab of this unit asks the real car**: you run both agents on CARLA's Monza, record each car's own cross-track error, and compare with racekit's predictions at lags 0, 1 and 2.
If CARLA's Unit 4 RMS is below its Unit 5 RMS, the real car behaves like a small delay, and you keep Unit 4's law; otherwise you keep pure pursuit.
Unit 8 measures the delay directly.

Next unit: grip.
So far the car could take any corner at any speed; Unit 6's dynamic car cannot.